In [1]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-03-02 09:41:41--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 108.157.79.67, 108.157.79.46, 108.157.79.95, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|108.157.79.67|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M  2.13MB/s    in 3m 10s  

2026-03-02 09:44:52 (1.55 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



In [ ]:
!head fhvhv_tripdata_2021-01.parquet

#### I Transformed the parquet file to  csv file

In [4]:
df = spark.read\
     .option('header','true')\
     .csv('fhvhv_tripdata_2021-01.csv')

In [ ]:
df.show()

In [20]:
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv

In [6]:
import pandas as pd

In [7]:
df_pandas = pd.read_csv('head.csv')

In [8]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
originating_base_num        str
request_datetime            str
on_scene_datetime           str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
trip_miles              float64
trip_time                 int64
base_passenger_fare     float64
tolls                   float64
bcf                     float64
sales_tax               float64
congestion_surcharge    float64
airport_fee             float64
tips                    float64
driver_pay              float64
shared_request_flag         str
shared_match_flag           str
access_a_ride_flag          str
wav_request_flag            str
wav_match_flag              str
dtype: object

In [9]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', StringType(), True), StructField('on_scene_datetime', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag', StringType(

In [10]:
from pyspark.sql import types

In [11]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('originating_base_num', types.StringType(), True), 
    types.StructField('request_datetime', types.TimestampType(), True), 
    types.StructField('on_scene_datetime', types.TimestampType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('trip_miles', types.DoubleType(), True), 
    types.StructField('trip_time', types.IntegerType(), True), 
    types.StructField('base_passenger_fare', types.DoubleType(), True), 
    types.StructField('tolls', types.DoubleType(), True), 
    types.StructField('bcf', types.DoubleType(), True), 
    types.StructField('sales_tax', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True), 
    types.StructField('airport_fee', types.DoubleType(), True), 
    types.StructField('tips', types.DoubleType(), True), 
    types.StructField('driver_pay', types.DoubleType(), True), 
    types.StructField('shared_request_flag', types.StringType(), True), 
    types.StructField('shared_match_flag', types.StringType(), True), 
    types.StructField('access_a_ride_flag', types.StringType(), True), 
    types.StructField('wav_request_flag', types.StringType(), True), 
    types.StructField('wav_match_flag', types.StringType(), True)
])


### Adding The schema to the csv file 

In [12]:
df = spark.read\
    .option('header','true')\
    .schema(schema)\
    .csv('fhvhv_tripdata_2021-01.csv')

In [16]:
df = df.repartition(24)

### Converting the csv file to parquet file using Spark

In [ ]:
df.write.parquet('fhvhv/2021/01')

### Reading to parquet file using Spark

In [13]:
df = spark.read.parquet('fhvhv/2021/01')

### Using Spark printSchema to display the table schema

In [14]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp (nullable = true)
 |-- on_scene_datetime: timestamp (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: integer (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_flag: st

### Using Spark to select the column from the parquet file adding a filter

In [15]:
df.select('hvfhs_license_num','PULocationID','DOLocationID','trip_miles','trip_time')\
  .filter(df.hvfhs_license_num == 'HV0003')\
  .show()

+-----------------+------------+------------+----------+---------+
|hvfhs_license_num|PULocationID|DOLocationID|trip_miles|trip_time|
+-----------------+------------+------------+----------+---------+
|           HV0003|          90|         249|      1.12|      287|
|           HV0003|          33|         224|      5.32|      674|
|           HV0003|          18|         185|      2.74|      537|
|           HV0003|          47|          18|      1.26|      382|
|           HV0003|          18|          51|       4.2|     1090|
|           HV0003|           4|         127|     12.28|     1487|
|           HV0003|         114|         148|       0.9|      507|
|           HV0003|         177|         177|      0.43|      251|
|           HV0003|          61|          76|       3.4|      831|
|           HV0003|         204|          23|      9.64|     1023|
|           HV0003|         241|         136|      1.63|      425|
|           HV0003|          53|          82|      5.69|      

In [16]:
from pyspark.sql import functions as F

## Adding column and selecting columns 

In [17]:
df\
    .withColumn('pickup_date',F.to_date(df.pickup_datetime))\
    .withColumn('dropoff_date',F.to_date(df.dropoff_datetime))\
    .withColumn('Alter',df.DOLocationID / 2 )\
    .select('pickup_date','dropoff_date','DOLocationID','Alter','trip_miles','trip_time')\
    .show()

+-----------+------------+------------+-----+----------+---------+
|pickup_date|dropoff_date|DOLocationID|Alter|trip_miles|trip_time|
+-----------+------------+------------+-----+----------+---------+
| 2021-01-02|  2021-01-02|         249|124.5|      1.12|      287|
| 2021-01-01|  2021-01-01|         247|123.5|     4.264|     1304|
| 2021-01-02|  2021-01-02|         224|112.0|      5.32|      674|
| 2021-01-01|  2021-01-01|         185| 92.5|      2.74|      537|
| 2021-01-02|  2021-01-02|          18|  9.0|      1.26|      382|
| 2021-01-02|  2021-01-02|          51| 25.5|       4.2|     1090|
| 2021-01-01|  2021-01-01|         127| 63.5|     12.28|     1487|
| 2021-01-01|  2021-01-01|         148| 74.0|       0.9|      507|
| 2021-01-01|  2021-01-01|         177| 88.5|      0.43|      251|
| 2021-01-01|  2021-01-01|          76| 38.0|       3.4|      831|
| 2021-01-02|  2021-01-02|          23| 11.5|      9.64|     1023|
| 2021-01-01|  2021-01-01|         145| 72.5|     10.73|     1

### Defining our own function

In [18]:
def transform(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0 :
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'
    
        

In [19]:
transform_udf = F.udf(transform, returnType=types.StringType())

In [20]:
df\
    .withColumn('pickup_date',F.to_date(df.pickup_datetime))\
    .withColumn('dropoff_date',F.to_date(df.dropoff_datetime))\
    .withColumn('Alter',df.DOLocationID / 2 )\
    .withColumn('base_id',transform_udf(df.dispatching_base_num))\
    .select('pickup_date','dropoff_date','DOLocationID','Alter','trip_miles','trip_time','base_id')\
    .show()

[Stage 4:>                                                          (0 + 1) / 1]

+-----------+------------+------------+-----+----------+---------+-------+
|pickup_date|dropoff_date|DOLocationID|Alter|trip_miles|trip_time|base_id|
+-----------+------------+------------+-----+----------+---------+-------+
| 2021-01-02|  2021-01-02|         249|124.5|      1.12|      287|  e/b38|
| 2021-01-01|  2021-01-01|         247|123.5|     4.264|     1304|  e/9ce|
| 2021-01-02|  2021-01-02|         224|112.0|      5.32|      674|  e/b47|
| 2021-01-01|  2021-01-01|         185| 92.5|      2.74|      537|  e/b35|
| 2021-01-02|  2021-01-02|          18|  9.0|      1.26|      382|  e/acc|
| 2021-01-02|  2021-01-02|          51| 25.5|       4.2|     1090|  e/b42|
| 2021-01-01|  2021-01-01|         127| 63.5|     12.28|     1487|  e/b38|
| 2021-01-01|  2021-01-01|         148| 74.0|       0.9|      507|  e/a39|
| 2021-01-01|  2021-01-01|         177| 88.5|      0.43|      251|  a/b43|
| 2021-01-01|  2021-01-01|          76| 38.0|       3.4|      831|  s/acd|
| 2021-01-02|  2021-01-02